In [19]:
# Import Libraries & Load Data

# Import core libraries for data manipulation and analysis
import pandas as pd
import numpy as np

# Load the training dataset
train_df = pd.read_csv('../data/train.csv')

In [20]:
# Feature Engineering & Data Cleaning

# Define a helper function to categorize family size into logical groups
def group_family(size):
    if size == 1:
        return 'Alone'
    elif size <= 4:
        return 'Small'
    else:
        return 'Large'

# --- 1. Title Extraction ---
# Extract titles from raw name string
train_df['Title'] = train_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Group rare titles into a single category
train_df['Title'] = train_df['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
train_df['Title'] = train_df['Title'].replace('Mlle', 'Miss')
train_df['Title'] = train_df['Title'].replace('Ms', 'Miss')
train_df['Title'] = train_df['Title'].replace('Mme', 'Mrs')

# Map titles to distinct numerical values
title_mapping = {"Mr": 1, "Miss": 2, "Mrs": 3, "Master": 4, "Rare": 5}
train_df['Title'] = train_df['Title'].map(title_mapping)
train_df['Title'] = train_df['Title'].fillna(0)

# --- 2. Family Grouping ---
# Calculate family size and map into Alone, Small, or Large groups
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1
train_df['FamilyGroup'] = train_df['FamilySize'].apply(group_family)

# Encode family groups numerically based on non-linear survival logic
family_mapping = {'Small': 1, 'Alone': 2, 'Large': 3}
train_df['FamilyGroup'] = train_df['FamilyGroup'].map(family_mapping)

# --- 3. Handling Missing Values & Encoding ---
# Fill missing age values with median
train_df['Age'] = train_df['Age'].fillna(train_df['Age'].median())

# Fill missing embarkation values with mode
train_df['Embarked'] = train_df['Embarked'].fillna(train_df['Embarked'].mode()[0])

# Map categorical text features to numerical labels
train_df['Sex'] = train_df['Sex'].map({'male': 0, 'female': 1})
train_df['Embarked'] = train_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

In [21]:
# Feature Selection & Data Split

from sklearn.model_selection import train_test_split

# Select engineered and cleaned features for the model
features = ['Pclass', 'Sex', 'Age', 'Embarked', 'Title', 'FamilyGroup']
X = train_df[features]
y = train_df['Survived']

# Split data into training and validation sets (80% train, 20% validation)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
# Feature Scaling

from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit the scaler on training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Transform validation data using the same scaler to avoid data leakage
X_val_scaled = scaler.transform(X_val)

In [23]:
# Hyperparameter Tuning & Model Training (GridSearchCV)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Define parameter grid for k-NN optimization
param_grid = {
    'n_neighbors': [5, 7, 9, 11, 13, 15, 17, 19, 21, 23],
    'weights': ['uniform', 'distance'],
    'p': [1, 2] # 1: Manhattan Distance, 2: Euclidean Distance
}

# Initialize the base k-NN model
knn_base = KNeighborsClassifier()

# Configure GridSearchCV with 5-fold cross-validation
grid_search = GridSearchCV(estimator=knn_base, param_grid=param_grid, cv=5, scoring='accuracy')

# Execute grid search on scaled training data
print("Searching for the best parameter combination...")
grid_search.fit(X_train_scaled, y_train)

# Display optimization results
print("\n--- Grid Search Results ---")
print(f"Best Parameters: {grid_search.best_params_}")
print("-" * 30)

# Evaluate the optimized model on the validation set
best_knn = grid_search.best_estimator_
y_pred = best_knn.predict(X_val_scaled)

# Calculate final validation accuracy
accuracy = accuracy_score(y_val, y_pred)
print(f"Final Validation Accuracy: {accuracy * 100:.2f}%")

Searching for the best parameter combination...

--- Grid Search Results ---
Best Parameters: {'n_neighbors': 19, 'p': 1, 'weights': 'uniform'}
------------------------------
Final Validation Accuracy: 82.68%


In [24]:
# Final Prediction & Kaggle Submission

# Load the unlabelled testing dataset
test_df = pd.read_csv('../data/test.csv')

# --- Replicate Feature Engineering on Test Data ---
test_df['Title'] = test_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
test_df['Title'] = test_df['Title'].replace(['Lady', 'Countess','Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
test_df['Title'] = test_df['Title'].replace('Mlle', 'Miss')
test_df['Title'] = test_df['Title'].replace('Ms', 'Miss')
test_df['Title'] = test_df['Title'].replace('Mme', 'Mrs')
test_df['Title'] = test_df['Title'].map(title_mapping)
test_df['Title'] = test_df['Title'].fillna(0)

test_df['FamilySize'] = test_df['SibSp'] + test_df['Parch'] + 1
test_df['FamilyGroup'] = test_df['FamilySize'].apply(group_family)
test_df['FamilyGroup'] = test_df['FamilyGroup'].map(family_mapping)

# --- Replicate Cleaning & Encoding on Test Data ---
test_df['Age'] = test_df['Age'].fillna(train_df['Age'].median()) 
test_df['Sex'] = test_df['Sex'].map({'male': 0, 'female': 1})
test_df['Embarked'] = test_df['Embarked'].fillna(train_df['Embarked'].mode()[0])
test_df['Embarked'] = test_df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# Select identical features and apply the fitted scaler
X_test_final = test_df[features]
X_test_scaled = scaler.transform(X_test_final)

# Predict survival utilizing the best k-NN model
predictions = best_knn.predict(X_test_scaled)

# Construct submission dataframe matching Kaggle formatting specifications
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': predictions
})

# Export predictions to CSV format
submission.to_csv('submission.csv', index=False)
print("Prediction completed! 'submission.csv' has been generated successfully.")
print("-" * 45)
print(submission.head(10))

Prediction completed! 'submission.csv' has been generated successfully.
---------------------------------------------
   PassengerId  Survived
0          892         0
1          893         1
2          894         0
3          895         0
4          896         1
5          897         0
6          898         1
7          899         0
8          900         1
9          901         0
